# Lab10 - Intro to PyTorch

In [22]:
import torch
import math

torch.cuda.is_available()

True

## Ex. 1) Batch Normalization

Define a tensor of shape $[B, 3, H, W]$ filled with random values to simulate a batch of RGB images.

**Steps to perform channel-wise batch normalization**:

1. Compute the **mean per color channel** by averaging over the batch and spatial dimensions
2. Compute the **standard deviation per color channel** over the same dimensions
3. Inspect the computed statistics to ensure they correspond to the RGB channels
    - print("Mean per channel:", mean)
    - print("Std per channel:", std)


4. Reshape the mean and standard deviation tensors to make them compatible with broadcasting
    - Verify the reshaped dimensions:
        - print("Reshaped mean:", mean.shape)
        - print("Reshaped std:", std.shape)


5. Apply channel-wise centering and normalization using broadcasting

6. Verify the result by recomputing the mean and standard deviation of the normalized batch
7. Check that the normalized images have approximately zero mean and unit variance per channel

    - print("New mean per channel:", new_mean)
    - print("New std per channel:", new_std)

In [2]:
B, H, W = 8, 64, 64

images = torch.rand(B, 3, H, W)

print(images.shape)

torch.Size([8, 3, 64, 64])


In [21]:
# Compute mean and std
mean = images.mean(dim=(0, 2, 3))
std = images.std(dim=(0, 2, 3))

print(f"Mean per channel: {mean}")
print(f"Std per channel: {std}")

# Apply channel-wise normalization
# This requires reshaping
mean = mean.reshape(1, 3, 1, 1)
std = std.reshape(1, 3, 1, 1)

norm_images = (images - mean) / std

norm_mean = norm_images.mean(dim=(0, 2, 3))
norm_std = norm_images.std(dim=(0, 2, 3))

print(f"Mean per channel: {norm_mean}")
print(f"Std per channel: {norm_std}")

Mean per channel: tensor([0.4977, 0.5012, 0.5023])
Std per channel: tensor([0.2898, 0.2901, 0.2877])
Mean per channel: tensor([ 8.9058e-09,  5.7975e-08, -3.8766e-08])
Std per channel: tensor([1., 1., 1.])


## Ex. 2) Multi-Head Attention

- Implement a **mini multi-head attention block** *from scratch* using only:
    - tensor reshaping (`view`, `reshape`, `transpose`, `permute`)
    - batched matrix multiplication (`matmul`)
    - broadcasting

In [45]:
B = 4 # Batch Size
T = 16 # Sequence Length (in Tokens)
D = 64 # Embeddings Size
H = 8
d = D // H

# Create the input
X = torch.randn(B, T, D, requires_grad=True)

Wq = torch.randn(D, D, requires_grad=True)
Wk = torch.randn(D, D, requires_grad=True)
Wv = torch.randn(D, D, requires_grad=True)
Wo = torch.randn(D, D, requires_grad=True)

# Compute Q, K, V
Q = X @ Wq # [B, T, D] @ [D, D] = [B, T, D]
K = X @ Wk
V = X @ Wv
print("Q, K, V:", Q.shape, K.shape, V.shape)

# Split into heads
Qh = Q.view(B, T, H, d).transpose(1, 2) # [B, T, D] -> [B, T, H, d] -> [B, H, T, d]
Kh = K.view(B, T, H, d).transpose(1, 2)
Vh = V.view(B, T, H, d).transpose(1, 2)
print("Qh, Kh, Vh:", Qh.shape, Kh.shape, Vh.shape)

# Compute Attentions
QK = Qh @ Kh.transpose(2, 3) # [B, H, T, d] @ [B, H, d, T] = [B, H, T, T]
S = QK / math.sqrt(d) # [B, H, T, T]
print("Scores S:", S.shape)

A = torch.softmax(S, dim=3) # [B, H, T, T]
print("Attention weights A:", A.shape)

Oh = A @ Vh # [B, H, T, T] @ [B, H, T, d] = [B, H, T, d]
print("Oh:", Oh.shape)

# Merge heads back
O = Oh.transpose(1, 2).reshape(B, T, D) # [B, T, D]
print("Merged out:", O.shape)

# Final Layer
Y = O @ Wo # [B, T, D] @ [D, D] = [B, T, D]
print("y:", Y.shape)

Q, K, V: torch.Size([4, 16, 64]) torch.Size([4, 16, 64]) torch.Size([4, 16, 64])
Qh, Kh, Vh: torch.Size([4, 8, 16, 8]) torch.Size([4, 8, 16, 8]) torch.Size([4, 8, 16, 8])
Scores S: torch.Size([4, 8, 16, 16])
Attention weights A: torch.Size([4, 8, 16, 16])
Oh: torch.Size([4, 8, 16, 8])
Merged out: torch.Size([4, 16, 64])
y: torch.Size([4, 16, 64])
